# MathTutor AI — GPU Training on Google Colab
**Before running:** Runtime → Change runtime type → T4 GPU


In [ ]:
# ── CELL 1: Verify GPU ────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "No GPU! Go to Runtime → Change runtime type → T4 GPU"
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"   PyTorch: {torch.__version__}")

In [ ]:
# ── CELL 2: Mount Google Drive (saves checkpoint here) ────────
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/MathSolverr/checkpoints', exist_ok=True)
print("✅ Drive mounted — checkpoints will be saved to My Drive/MathSolverr/checkpoints/")

In [ ]:
# ── CELL 3: Clone your project from GitHub ────────────────────
# Replace with your actual GitHub repo URL after you push
GITHUB_URL = "https://github.com/YOUR_USERNAME/MathSolverr.git"  # ← change this

!git clone {GITHUB_URL} /content/MathSolverr
%cd /content/MathSolverr
print("✅ Project cloned")

In [ ]:
# ── CELL 3b: Alternative — upload a zip instead of GitHub ─────
# (Run this cell ONLY if you didn't use Cell 3 above)
#
# from google.colab import files
# uploaded = files.upload()   # upload MathSolverr.zip
# !unzip MathSolverr.zip -d /content/
# %cd /content/MathSolverr
print("Skipped — using GitHub clone")

In [ ]:
# ── CELL 4: Install dependencies ──────────────────────────────
!pip install -q \
    datasets==5.0.1 \
    sympy==1.14.0 \
    pyyaml \
    python-multipart \
    fastapi \
    uvicorn \
    editdistance \
    Pillow

# Torch is pre-installed on Colab with CUDA support
import torch
print(f"✅ All deps installed | CUDA: {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── CELL 5: Set HF cache to Drive (avoids re-downloading) ─────
import os
os.environ['HF_DATASETS_CACHE'] = '/content/drive/MyDrive/MathSolverr/hf_cache'
os.environ['HF_HOME']           = '/content/drive/MyDrive/MathSolverr/hf_cache'
os.makedirs(os.environ['HF_DATASETS_CACHE'], exist_ok=True)
print("✅ HF cache → Drive (dataset won't re-download next session)")

In [ ]:
# ── CELL 6: Configure training ─────────────────────────────────
# Tweak these to your preference
EPOCHS        = 50
BATCH_SIZE    = 32      # T4 comfortably handles 32; use 64 for A100
TRAIN_SUBSET  = 50000   # None = full dataset
VAL_SUBSET    = 5000
LR            = 1e-4

print(f"Training config:")
print(f"  Epochs:       {EPOCHS}")
print(f"  Batch size:   {BATCH_SIZE}")
print(f"  Train subset: {TRAIN_SUBSET}")
print(f"  Val subset:   {VAL_SUBSET}")

In [ ]:
# ── CELL 7: Train ─────────────────────────────────────────────
# Note: Checkpoints are saved to checkpoints/ after each completed epoch.
!python train.py \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --train-subset {TRAIN_SUBSET} \
    --val-subset {VAL_SUBSET}


In [ ]:
# ── CELL 8: Copy checkpoint to Google Drive ───────────────────
import shutil, os

ckpt_name = None
for name in ['best_model.pt', 'latest_model.pt']:
    if os.path.exists(f'/content/MathSolverr/checkpoints/{name}'):
        ckpt_name = name
        break

if ckpt_name:
    src = f'/content/MathSolverr/checkpoints/{ckpt_name}'
    dst = '/content/drive/MyDrive/MathSolverr/checkpoints/best_model.pt'
    shutil.copy(src, dst)
    size_mb = os.path.getsize(dst) / 1e6
    print(f'✅ Checkpoint ({ckpt_name}) saved to Drive: {dst} ({size_mb:.1f} MB)')
else:
    print('❌ No checkpoint found in /content/MathSolverr/checkpoints/')
    print('   Did Cell 7 finish at least 1 epoch? If interrupted with ^C, run it again.')


In [ ]:
# ── CELL 9: Download checkpoint directly to your PC ───────────
from google.colab import files
import os
downloaded = False
for name in ['best_model.pt', 'latest_model.pt']:
    path = f'/content/MathSolverr/checkpoints/{name}'
    if os.path.exists(path):
        files.download(path)
        print(f'✅ Downloaded {name}')
        downloaded = True
        break
if not downloaded:
    print('❌ No checkpoint file available to download.')


In [ ]:
# ── CELL 10: Run evaluation on Colab ──────────────────────────
!python eval.py --checkpoint checkpoints/best_model.pt

## After downloading the checkpoint
Copy `best_model.pt` into `d:\MathSolverr\checkpoints\` on your PC,
then start the app as normal:
```
python run.py
```
The server auto-loads the checkpoint on startup.


---
## Connect VS Code to this Colab session
Run the cell below to open an SSH tunnel, then connect VS Code via Remote SSH.


In [ ]:
# ── CELL 11: VS Code SSH tunnel (optional) ────────────────────
# This lets you edit files and run terminals inside this Colab GPU session
# from VS Code on your PC.

!pip install -q colab-ssh

from colab_ssh import launch_ssh_cloudflared, init_git_globally

# Set a password for the SSH connection
launch_ssh_cloudflared(password="mathtutor123")

# Instructions printed by the cell:
# 1. Install the VS Code extension: "Remote - SSH" (ms-vscode-remote.remote-ssh)
# 2. In VS Code: Ctrl+Shift+P → "Remote-SSH: Connect to Host"
# 3. Enter the hostname shown above (looks like: xxxx.trycloudflare.com)
# 4. Username: root  |  Password: mathtutor123
# 5. Open folder: /content/MathSolverr